# **🏢 Amazon ML Challenge 2026 - Business Entity Resolution**

This project focuses on matching business entity records across noisy, independent data sources with no shared unique IDs.

In this notebook:

# **🧹 01. Data Audit, Cleaning & Normalization Pipeline**

This notebook handles data ingestion, structural verification, noise distribution analysis, and text standardization across Source 1 (reference), Source 2, and Source 3 before candidate generation (blocking).

## **1. Problem Definition**

Business identity data arrives from disparate sources with typos, legal suffixes, missing street details, and varying regional address formats.

Given raw business records across 3 TSV files, normalize text components so that downstream blocking rules can group candidate pairs accurately without dropping true matches.



## **2. Data Architecture & Column Mapping**

All datasets are tab-separated (`.tsv`) to safely handle commas inside address strings.

**Key Datasets:**
- `train_source1.tsv`: Reference deduplicated records (`S1-` prefix)
- `train_source2.tsv`: Query/fragment source 1 (`S2-` prefix)
- `train_source3.tsv`: Query/fragment source 2 (`S3-` prefix)
- `train_ground_truth.tsv`: Ground truth mapping (`source1_entity_id` → matched `S2`/`S3` IDs)

**Schema Across All Source Files:**

| **Column** | **Type** | **Description** | **Common Noise / Challenges** |
|---|---|---|---|
| `entity_id` | String | Unique record ID (`S1-xxx`, `S2-xxx`, `S3-xxx`) | Source prefix indicator |
| `business_name` | String | Name of the business | Legal suffixes (LLC, Pvt Ltd, Inc), typos, word reordering |
| `business_address` | String | Street address / location | Missing values, landmark references, abbreviations (`Rd`, `St`) |
| `country` | String | Country code (`US`, `India` in Train; `France` in Test) | Domain shift across regions |



## **3. Pipeline Execution Steps**

1. TSV Integrity & Ingestion: Verify tab separation and memory usage.
2. Missingness & Country Coverage Audit: Check null percentages for names and addresses across US vs India.
3. String Normalization Engine: Lowercasing, punctuation stripping, whitespace collapsing, and legal suffix stripping.
4. Digit & Token Extraction: Isolate street numbers/zip codes and sorted name tokens for blocking keys.

```python
Raw TSV file
     |
     v
Step 1: Does it load correctly? (integrity check)
     |
     v
Step 2: What's missing, and does it differ by country?
     |
     v
Step 3: Clean text (same business should look the same)
     |
     v
Step 4: Pull out blocking ingredients (zip, number, name tokens)
     |
     v
Ready to actually build bucket logic
```

### **1. Loading and testing the dataset**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set up data directory path
DATA_DIR = Path("../data/student_resource/dataset/train")

# ---------------------------------------------------------
# STEP 1: TSV Integrity & Ingestion
# ---------------------------------------------------------
print("Loading datasets...")

df_s1 = pd.read_csv(DATA_DIR / "train_source1.tsv", sep="\t")
df_s2 = pd.read_csv(DATA_DIR / "train_source2.tsv", sep="\t")
df_s3 = pd.read_csv(DATA_DIR / "train_source3.tsv", sep="\t")
df_gt = pd.read_csv(DATA_DIR / "train_ground_truth.tsv", sep="\t")

print("Ingestion complete!\n")

Loading datasets...
Ingestion complete!

